# Stage 7 — Cleanup (prepare for next iteration)
Removes **disposable temporary state** so Stages 1–5 can run fresh, while
protecting the durable assets (pool, manifests, models).

**Safety:** this cell only REPORTS by default. Set `CONFIRM=True` to actually
delete. Clearing `incoming/` is a separate opt-in because unmerged images
there would be lost.

In [1]:
import os, shutil
DATASET_ROOT='/dataset'

# ---- SET THESE ----
CONFIRM        = True   # must be True to actually delete anything
CLEAR_CROPS    = True    # cached crops — safe, regenerable from pool+manifest
CLEAR_LS_FILES = True    # Label Studio import/export hand-off files
CLEAR_INCOMING = True   # DANGER: only if incoming images are already merged
# -------------------

# NEVER touched: pool/, manifests/, models/  (durable assets)
PROTECTED = ['pool','manifests','models']

In [2]:
# report sizes of deletable targets
def dir_size(p):
    total=0
    for dp,_,fs in os.walk(p):
        for f in fs:
            try: total+=os.path.getsize(os.path.join(dp,f))
            except OSError: pass
    return total
def human(n): 
    for u in ['B','KB','MB','GB','TB']:
        if n<1024: return f'{n:.1f}{u}'
        n/=1024
    return f'{n:.1f}PB'

targets=[]
if CLEAR_CROPS:    targets.append(('crops',      os.path.join(DATASET_ROOT,'crops')))
if CLEAR_LS_FILES: targets.append(('ls_import',  os.path.join(DATASET_ROOT,'ls_import')))
if CLEAR_LS_FILES: targets.append(('ls_export',  os.path.join(DATASET_ROOT,'ls_export')))
if CLEAR_INCOMING: targets.append(('incoming',   os.path.join(DATASET_ROOT,'incoming')))

print('=== cleanup plan ===')
for name,path in targets:
    if os.path.isdir(path):
        print(f'  DELETE  {name:12s} {human(dir_size(path)):>10}  {path}')
    else:
        print(f'  (absent) {name:12s}            {path}')
print('\n=== protected (never deleted) ===')
for p in PROTECTED:
    fp=os.path.join(DATASET_ROOT,p)
    if os.path.isdir(fp): print(f'  KEEP    {p:12s} {human(dir_size(fp)):>10}')
print('\nCONFIRM =',CONFIRM,'(set True to execute)')

=== cleanup plan ===
  DELETE  crops             4.7GB  /dataset/crops
  DELETE  ls_import        22.6KB  /dataset/ls_import
  DELETE  ls_export        93.0KB  /dataset/ls_export
  DELETE  incoming         53.6MB  /dataset/incoming

=== protected (never deleted) ===
  KEEP    pool             26.1GB
  KEEP    manifests         8.0MB
  KEEP    models            4.5GB

CONFIRM = True (set True to execute)


In [3]:
# safety check before clearing incoming: verify images are in the pool
if CLEAR_INCOMING:
    import sys; sys.path.insert(0,'/workspace/lib')
    import dataset_lib as D
    inc=os.path.join(DATASET_ROOT,'incoming')
    pool_ids=set(D.list_pool(DATASET_ROOT))
    unmerged=[]
    if os.path.isdir(inc):
        for f in os.listdir(inc):
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                iid=D.image_id(os.path.join(inc,f))
                if iid not in pool_ids: unmerged.append(f)
    if unmerged:
        print(f'WARNING: {len(unmerged)} incoming images are NOT in the pool:')
        for f in unmerged[:10]: print('  ',f)
        print('These would be LOST. Verify/merge them before clearing incoming.')
    else:
        print('All incoming images are present in the pool — safe to clear.')
else:
    print('CLEAR_INCOMING is False — incoming/ will not be touched.')

   BR_20100702_220746.JPG
   BR_20100711_225420.JPG
   BR_20100722_233308.JPG
   BR_20160104_200730.JPG
   BR_20160210_180304.JPG
   BR_20160503_235306.JPG
   BR_20160530_214930.JPG
   BR_20160617_013131.JPG
   BR_20160617_070822.JPG
   BR_20160617_190946.JPG
These would be LOST. Verify/merge them before clearing incoming.


In [4]:
# execute deletion — rebuilds targets from CURRENT flags so it can't use a stale list
if not CONFIRM:
    print('CONFIRM is False — nothing deleted. Set CONFIRM=True and re-run this cell.')
else:
    to_delete=[]
    if CLEAR_CROPS:    to_delete.append(os.path.join(DATASET_ROOT,'crops'))
    if CLEAR_LS_FILES: to_delete += [os.path.join(DATASET_ROOT,'ls_import'), os.path.join(DATASET_ROOT,'ls_export')]
    if CLEAR_INCOMING: to_delete.append(os.path.join(DATASET_ROOT,'incoming'))
    for path in to_delete:
        if os.path.isdir(path):
            shutil.rmtree(path); print('deleted',path)
        else:
            print('(absent, skipped)',path)
    for d in ['incoming','ls_import','ls_export']:
        os.makedirs(os.path.join(DATASET_ROOT,d),exist_ok=True)
    print('cleanup complete. pool/ manifests/ models/ untouched.')

deleted /dataset/crops
deleted /dataset/ls_import
deleted /dataset/ls_export
deleted /dataset/incoming
cleanup complete. pool/ manifests/ models/ untouched.


Ready for the next iteration: drop new images in `incoming/` and run Stage 1.
Note: `crops/` will be regenerated by Stage 4's crop cell on the next retrain.